**Cell 1: Environment & Setup**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import sklearn
import os
import yfinance as yf
import pandas_datareader.data as web
from google.colab import drive
from datetime import datetime
from scipy.stats import norm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Mount Drive & Setup Folders
drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/MSFin_Python'
sub_folders = ['data', 'notebooks', 'output']

for folder in sub_folders:
    path = os.path.join(base_dir, folder)
    os.makedirs(path, exist_ok=True)

print("-" * 40)
print(f"SUCCESS: Environment Ready at {base_dir}")
print("-" * 40)

**Cell 2: Data Engine (Levels & Returns)**

In [ ]:
# --- CONFIGURATION ---
ASSETS = ['SPY', 'QQQ', 'RSP', 'GLD', 'IEF']
MACRO = ['VIXCLS', 'BAMLH0A0HYM2']
START_DATE = "2005-01-01"

# --- FETCH DATA ---
print("Fetching Data...")
# We pull price levels for time series
prices = yf.download(ASSETS, start=START_DATE, auto_adjust=True)['Close']
macro = web.DataReader(MACRO, 'fred', START_DATE)
macro.columns = ['VIX', 'HYOAS']

# Master Levels DF
df_levels = prices.join(macro, how='inner').dropna()

# Master Returns DF (for Histograms and Dot Plots)
df_returns = df_levels.copy()
for col in ASSETS:
    df_returns[col] = df_returns[col].pct_change() * 100
df_returns = df_returns.dropna()# 1. TIME SERIES (LEVELS)
fig, axes = plt.subplots(len(df_levels.columns), 1, figsize=(12, 20), sharex=True)

for i, col in enumerate(df_levels.columns):
    axes[i].plot(df_levels.index, df_levels[col], color='navy', lw=1.5)
    axes[i].set_title(f'Historical Level: {col}', fontweight='bold', loc='left')
    axes[i].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'output', '01_eda_levels.png'))
plt.show()# 2. HISTOGRAMS (RETURNS/LEVELS)
fig, axes = plt.subplots(len(df_returns.columns), 1, figsize=(10, 20))

for i, col in enumerate(df_returns.columns):
    sns.histplot(df_returns[col], kde=True, ax=axes[i], color='darkred', bins=50)
    axes[i].set_title(f'Distribution: {col}', fontweight='bold')
    # Labeling appropriately
    unit = "% Weekly Return" if col in ASSETS else "Index Level"
    axes[i].set_xlabel(unit)

plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'output', '02_eda_distributions.png'))
plt.show()

**Cell 3: Time Series (Levels Only)**

In [ ]:
# 3. DOT PLOTS (SPY as TARGET)
target_vars = ['QQQ', 'RSP', 'GLD', 'IEF', 'VIX', 'HYOAS']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(target_vars):
    sns.regplot(x=df_returns[col], y=df_returns['SPY'], ax=axes[i],
                scatter_kws={'alpha':0.3, 'color':'gray'},
                line_kws={'color':'blue', 'lw':2})
    axes[i].set_title(f'SPY vs. {col}', fontweight='bold')
    axes[i].set_ylabel('SPY Weekly Return %')
    axes[i].set_xlabel(f'{col} Weekly Return/Level')

plt.tight_layout()
plt.savefig(os.path.join(base_dir, 'output', '03_eda_dotplots.png'))
plt.show()